# Verkehrszeichen-Export für radinfra.de

Schlankere Fassung von `radinfra_1_export.ipynb`.
Die Logik liegt in [`cw_campaign.py`](cw_campaign.py), geteilt mit
[`maproulette_tasks.ipynb`](maproulette_tasks.ipynb).
Tests: `pytest test_cw_campaign.py`.

> **`x_` läuft wöchentlich auf dem Server** (`scripts/run_mapillary_notebooks.sh`, Pipeline `ts`)
> und bleibt unverändert. `xb_` ist die Gegenprobe: gleiche Eingabe, gleiche Ausgabe.
> Erst wenn der Vergleich in der letzten Zelle sauber ist, lohnt sich der Umstieg —
> und der ist dann eine Zeile im Runner.

**Erzeugt drei Dateien in `ts_output/`**

| Datei | |
| --- | --- |
| `mapillary_trafficsigns_bicycle_latest.geojson.gz` | der eigentliche Export |
| `signs_by_month.svg` | Statistikgrafik für die README |
| `README.md` | Beschreibung des Ordners |

**Was gegenüber `x_` anders ist**

1. Zeichentabelle, Laden, Zeitfilter und Autobahn-Abstand kommen aus `cw_campaign` —
   dieselben Funktionen, die `1b_` nutzt. Die sieben Zeichendefinitionen standen
   bisher in `x_` und `1_` getrennt.
2. Der Autobahn-Ausschluss läuft über `distance_to_nearest` statt über einen
   gepufferten Zweitdatensatz plus `mark_intersections`.
3. `import mapillary as mly` ist raus — `x_` importiert die Library, benutzt sie aber nie.

Die Ausgabe soll **byteweise dieselbe** sein wie bei `x_`. Genau das prüft die letzte Zelle.

## Einstellungen

In [ ]:
# Kein %autoreload hier - anders als 1b_ läuft dieses Notebook unbeaufsichtigt
# per nbconvert auf dem Server. Eine IPython-Erweiterung, die dort nichts nützt,
# ist nur eine weitere Stelle, an der ein Lauf scheitern kann.
from pathlib import Path

import geopandas as gpd
import pandas as pd

# cw_campaign.py liegt eine Ebene höher, weil beide Cycleway-Kampagnen es nutzen.
# Das ".." funktioniert, weil Jupyter und nbconvert das Arbeitsverzeichnis auf das
# Notebook-Verzeichnis setzen — dieselbe Annahme wie bei ../../output/ und ../utils/.
import sys

sys.path.insert(0, "..")
import cw_campaign as cw

ordner_zeichen = Path("../../output")
pfad_metadaten = ordner_zeichen / "ml-ts_metadata.json"
pfad_autobahnen = Path("../utils/processed_motorways_germany_251215.parquet")  # ändert sich kaum
ordner_ausgabe = Path("ts_output")

# Für radinfra.de wird weiter zurückgeschaut als für die MapRoulette-Kampagne:
# hier zählt jede Sichtung, nicht nur dauerhaft stehende Schilder.
zuletzt_gesehen_nach = "2023-01-01"

# Nur die Spalten lesen, die gebraucht werden - der ts-Lauf hat auf dem Server 6 GB.
spalten = ["id", "value", "first_seen_at", "last_seen_at", "geometry"]

ordner_ausgabe.mkdir(parents=True, exist_ok=True)
print("Zeichen:", ", ".join(sorted({code for code, _ in cw.ZEICHEN.values()})))

## 1 · Verkehrszeichen einlesen

Der Vollständigkeits-Guard ist der Grund, warum hier ein Abbruch steht und keine
Warnung: fehlt eine Bundesland-Datei, fehlt sie auch im publizierten Ergebnis.
Am 26.08.2026 ist genau das still durchgelaufen.

In [ ]:
# Sollwert aus dem Tile-Cache; ohne Tile-Cache (nur gespiegelte Parquets) None.
erwartet = cw.count_expected_states("../../prep/tile_cache/DE-*_tiles.json")
print("Tile-Cache kennt", erwartet, "Bundesländer")

signs = cw.load_features(
    ordner_zeichen, values=cw.ZEICHEN, columns=spalten, expect_files=erwartet
)
signs.head()

In [ ]:
ml_data_from, processed_date, bundeslaender = cw.read_dataset_metadata(pfad_metadaten)
print("ml_data_from: ", ml_data_from)
print("processed_date:", processed_date)
print("Bundesländer:  ", len(bundeslaender))

## 2 · Filter

In [ ]:
# Zeitfilter. min_months=0, weil hier anders als in 1b_ keine Mindeststandzeit gilt.
signs = cw.filter_stable_signs(signs, zuletzt_gesehen_nach, min_months=0)

In [ ]:
# Schilder an Autobahnen raus - dort sind die Erkennungen überwiegend falsch positiv.
autobahnen = gpd.read_parquet(pfad_autobahnen)
abstand_mw = cw.distance_to_nearest(signs, autobahnen, cw.AUTOBAHN_ABSTAND_M)
del autobahnen

# Kein reset_index: der Index wird beim Export zur Top-Level-id der Features.
signs = signs[abstand_mw == float("inf")]
print(f"nach Autobahn-Filter: {len(signs):,}".replace(",", "."))

## 3 · Beschriftung und Hinweise

In [ ]:
signs = cw.add_sign_labels(signs)
signs = cw.add_hinweis(signs)

export = signs[cw.EXPORT_SPALTEN]
print(export["Hinweis"].str.strip().ne("").sum(), "Zeichen mit Hinweistext")
export.head()

## 4 · Export schreiben

In [ ]:
pfad_geojson = ordner_ausgabe / "mapillary_trafficsigns_bicycle_latest.geojson.gz"

cw.write_geojson_gz(export, pfad_geojson)
cw.assert_ids_are_strings(pfad_geojson)

## 5 · Statistikgrafik

Reine Darstellung, deshalb bewusst im Notebook geblieben und nicht im Modul.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from matplotlib.ticker import FuncFormatter

sns.set_theme(style="whitegrid", context="notebook", rc={
    "axes.facecolor": "#e4e4ed",
    "figure.facecolor": "#DADADF",
    "grid.linestyle": ":",
    "grid.alpha": 0.7,
})

monat = pd.to_datetime(export["last_seen_at"]).dt.to_period("M").dt.to_timestamp()
pivot = export.groupby([monat.rename("month"), "traffic_sign_description"]).size().unstack(fill_value=0).sort_index()
pivot = pivot.reindex(pd.date_range(pivot.index.min(), pivot.index.max(), freq="MS"), fill_value=0)
pivot.index.name = "month"

# Reihenfolge der README-Tabelle, damit Grafik und Tabelle zusammenpassen.
# Gruppiert wird nach der deutschen Bezeichnung (so heißen die Spalten in
# den Daten), beschriftet mit der englischen - wie in der README-Tabelle.
reihenfolge = [(c, b, e) for c, _, b, e, *_ in cw.README_ZEICHEN if b in pivot.columns]
oben_nach_unten = [b for _, b, _ in reihenfolge]
farben = [plt.get_cmap("tab10")(i) for i in range(len(oben_nach_unten))]

fig, ax = plt.subplots(figsize=(14, 6))
pivot[oben_nach_unten[::-1]].plot(kind="bar", stacked=True, ax=ax, color=farben[::-1])

# Legende in der gewünschten Reihenfolge, VZ-Code monospaced ausgerichtet.
griffe = dict(zip(*reversed(ax.get_legend_handles_labels())))
breite = max(len(c) for c, _, _ in reihenfolge)
ax.legend(
    [griffe[b] for _, b, _ in reihenfolge],
    [f"{c.ljust(breite)}    {e}".replace(" ", "\u00A0") for c, _, e in reihenfolge],
    title="Traffic sign – bicycle infrastructure",
    bbox_to_anchor=(1.02, 1), loc="upper left",
    prop={"family": "monospace", "size": 9},
    handletextpad=1.5, labelspacing=0.7,
)

ax.set_xticks(range(len(pivot.index)))
ax.set_xticklabels([d.strftime("%Y-%m") for d in pivot.index], rotation=45, ha="right")
ax.tick_params(axis="x", labelsize=9)
ax.ticklabel_format(style="plain", axis="y")
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x:,.0f}"))
ax.yaxis.set_major_locator(mtick.MaxNLocator(integer=True))
ax.tick_params(axis="y", labelsize=10)
ax.grid(axis="y", linestyle=":", linewidth=0.8, alpha=0.6)
ax.set_xlabel("Year-Month")
ax.set_ylabel("Number of signs")
ax.set_title("Detected traffic signs per month\nBicycle infrastructure")

summen = pivot.sum(axis=1).to_numpy()
for i, summe in enumerate(summen):
    if summe > 0:
        ax.text(i, summe + max(summen) * 0.01, str(int(summe)), ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(ordner_ausgabe / "signs_by_month.svg", bbox_inches="tight")
plt.show()

## 6 · README schreiben

Die Delta-Spalte vergleicht mit `signs_history.json`. Die Datei liegt neben den
Outputs und wird mit ihnen nach B2 gespiegelt — `ts_output/` ist nicht in git,
ein Branch-Wechsel hat den Ordner schon einmal geleert. Fehlt die Historie,
bleibt die Spalte leer und der Lauf geht weiter.

In [ ]:
stand = cw.dataset_stand([ml_data_from, processed_date])
print("Datenstand für die README:", stand)

anzahl_je_zeichen = export["traffic_sign_description"].value_counts().to_dict()

# Vor dem Anhängen lesen, sonst wäre der Vergleichswert der aktuelle Lauf.
historie = cw.load_sign_history(ordner_ausgabe)
vorher, vorher_datum = cw.latest_sign_counts(historie, before=stand)
print("Vergleichslauf:", vorher_datum or "keiner - Delta-Spalte bleibt leer")

readme = cw.build_readme(
    anzahl_je_zeichen,
    stand,
    previous=vorher,
    previous_date=vorher_datum,
    seit=zuletzt_gesehen_nach,
    autobahn_abstand_m=int(cw.AUTOBAHN_ABSTAND_M),
)
(ordner_ausgabe / "README.md").write_text(readme, encoding="utf-8")
cw.append_sign_history(ordner_ausgabe, stand, anzahl_je_zeichen, runs=historie)
print(readme[:400])

## 7 · Gegenprobe gegen `x_`

Setzt einen Referenzlauf von `x_` in einem anderen Ordner voraus:

```bash
# x_ einmal laufen lassen, Ergebnis wegkopieren
cp -r ts_output ts_output_ref
```

Verglichen wird die entpackte GeoJSON — der gzip-Rohbytevergleich taugt nicht, weil
im gzip-Header ein Zeitstempel steckt und zwei Läufe sich darin immer unterscheiden.

In [ ]:
import gzip
import hashlib
import json

ordner_referenz = Path("ts_output_ref")

def geojson_fingerabdruck(pfad):
    with gzip.open(pfad, "rt", encoding="utf-8") as f:
        roh = f.read()
    daten = json.loads(roh)
    return {
        "sha256": hashlib.sha256(roh.encode("utf-8")).hexdigest(),
        "features": len(daten["features"]),
        "spalten": list(daten["features"][0]["properties"].keys()),
        "erste_id": daten["features"][0]["properties"]["id"],
        "letzte_id": daten["features"][-1]["properties"]["id"],
    }

if not ordner_referenz.exists():
    print(f"{ordner_referenz} fehlt - x_ laufen lassen und ts_output dorthin kopieren")
else:
    a = geojson_fingerabdruck(ordner_referenz / "mapillary_trafficsigns_bicycle_latest.geojson.gz")
    b = geojson_fingerabdruck(ordner_ausgabe / "mapillary_trafficsigns_bicycle_latest.geojson.gz")

    for schluessel in a:
        gleich = "OK  " if a[schluessel] == b[schluessel] else "NEIN"
        print(f"  {gleich} {schluessel}")
        if a[schluessel] != b[schluessel]:
            print(f"       x_:  {a[schluessel]}")
            print(f"       xb_: {b[schluessel]}")

    alt = (ordner_referenz / "README.md").read_text(encoding="utf-8")
    neu = (ordner_ausgabe / "README.md").read_text(encoding="utf-8")
    print(f"  {'OK  ' if alt == neu else 'NEIN'} README.md")

    # matplotlib schreibt in jede SVG einen Zeitstempel und würfelt die
    # clip-path-ids pro Prozess neu. Zwei Läufe von x_ unterscheiden sich darin
    # genauso - das ist kein inhaltlicher Unterschied, also beides wegnormieren.
    import re
    def svg_ohne_rauschen(pfad):
        s = pfad.read_text(encoding="utf-8")
        s = re.sub(r"<dc:date>[^<]*</dc:date>", "<dc:date/>", s)
        return re.sub(r"\b[a-z]p?[0-9a-f]{8,}\b", "ID", s)

    gleich = svg_ohne_rauschen(ordner_referenz / "signs_by_month.svg") == svg_ohne_rauschen(
        ordner_ausgabe / "signs_by_month.svg"
    )
    print(f"  {'OK  ' if gleich else 'NEIN'} signs_by_month.svg (ohne Zeitstempel und clip-path-ids)")